# SSL timbre augmentation exploration

Global transforms affect the complete clip. Local transforms are independently randomized over 1–4 overlapping chunks and recombined with interpolation. Every output has exactly the same shape as the input.

In [ ]:
from pathlib import Path
import random
import time

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display
from torch.utils.data import DataLoader, Dataset

from after.encoder_ssl.audio_augmentations import (
    DEFAULT_ECHOTHIEF_PATH, TimbreAugmentation,
    apply_local_in_chunks, benchmark_augmentation, compressor,
    loudness, pitch_shift, reverb, silence, time_stretch,pedal_pitchshift
)

random.seed(0)
np.random.seed(0)
print(f'EchoThief IRs: {len(list(DEFAULT_ECHOTHIEF_PATH.rglob("*.wav")))} files')

## Load an example
Set `AUDIO_PATH` to test a real recording. The fallback is a three-second harmonic signal, so the notebook works without another download.

In [ ]:
AUDIO_PATH = None  # e.g. '/path/to/example.wav'
SAMPLE_RATE = 44_100

if AUDIO_PATH is not None:
    audio, source_sr = sf.read(AUDIO_PATH, dtype='float32', always_2d=True)
    audio = audio.mean(axis=1)
    if source_sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=source_sr, target_sr=SAMPLE_RATE)
else:
    t = np.arange(3 * SAMPLE_RATE, dtype=np.float32) / SAMPLE_RATE
    audio = sum(np.sin(2 * np.pi * 220 * harmonic * t) / harmonic
                for harmonic in range(1, 7)).astype(np.float32) * 0.12

audio = np.asarray(audio, dtype=np.float32)
print(audio.shape, audio.dtype, f'{len(audio) / SAMPLE_RATE:.2f} seconds')
display(Audio(audio, rate=SAMPLE_RATE))

In [ ]:
def inspect_audio(name, transformed):
    assert transformed.shape == audio.shape
    figure, axes = plt.subplots(1, 2, figsize=(12, 3))
    axes[0].plot(np.arange(len(transformed)) / SAMPLE_RATE, transformed, linewidth=0.5)
    axes[0].set(title=name, xlabel='Time (s)', ylabel='Amplitude')
    spectrum = librosa.amplitude_to_db(
        np.abs(librosa.stft(transformed, n_fft=2048, hop_length=512)), ref=np.max)
    librosa.display.specshow(spectrum, sr=SAMPLE_RATE, hop_length=512,
                             x_axis='time', y_axis='log', ax=axes[1])
    axes[1].set(title=f'{name}: log-frequency spectrogram')
    plt.tight_layout()
    plt.show()
    display(Audio(transformed, rate=SAMPLE_RATE))

## Global augmentations, individually

In [ ]:
global_augmentations = {
    'compressor': compressor(p=1.0),
    'reverb': reverb(p=1.0),
}

global_results = {}
for name, transform in global_augmentations.items():
    global_results[name] = transform(audio.copy(), sample_rate=SAMPLE_RATE)
    inspect_audio(name, global_results[name])

## Local augmentations, individually
Each example below uses four independently augmented chunks with 50 ms of context on each side of an internal boundary.

In [ ]:
local_augmentations = {
    'pitch shift [-3, 3] semitones': pitch_shift(p=1.0),
    'loudness [-10, 0] dB': loudness(p=1.0),
    'time stretch [0.8, 1.2]': time_stretch(p=1.0),
    'silence': silence(p=1.0),
    'pitch shift pedalboard': pedal_pitchshift(p=1.),
}

local_results = {}
for name, transform in local_augmentations.items():
    local_results[name] = apply_local_in_chunks(
        audio, SAMPLE_RATE, [transform], num_chunks=4, overlap_seconds=0.1)
    inspect_audio(name, local_results[name])

## Combined callable

In [ ]:
combined_augmentation = TimbreAugmentation(
    compressor_p=0.5,
    reverb_p=0.5,
    pitch_shift_p=0.5,
    loudness_p=0.5,
    time_stretch_p=0.5,
    silence_p=0.25,
    min_chunks=1,
    max_chunks=4,
    overlap_seconds=0.05,
)

combined = combined_augmentation(audio, SAMPLE_RATE)
inspect_audio('combined augmentation', combined)

## Single-worker inference speed
Audiomentations runs on CPU. The real-time factor is processing time divided by audio duration; values below 1 mean faster than real time.

In [ ]:
benchmarks = {}
for name, transform in global_augmentations.items():
    benchmarks[f'global: {name}'] = benchmark_augmentation(
        transform, audio, SAMPLE_RATE, iterations=5)
for name, transform in local_augmentations.items():
    local_callable = lambda x, sr, transform=transform: apply_local_in_chunks(
        x, sr, [transform], num_chunks=10, overlap_seconds=0.05)
    benchmarks[f'local: {name}'] = benchmark_augmentation(
        local_callable, audio, SAMPLE_RATE, iterations=5)
benchmarks['combined'] = benchmark_augmentation(
    combined_augmentation, audio, SAMPLE_RATE, iterations=10)
pd.DataFrame(benchmarks).T.sort_values('seconds_per_example')

## DataLoader throughput
This approximates online augmentation. Compare workers, but remember that storage and the rest of your dataset pipeline also affect production throughput.

In [ ]:
class AugmentationBenchmarkDataset(Dataset):
    def __init__(self, source, transform, length=32):
        self.source = source
        self.transform = transform
        self.length = length

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        return self.transform(self.source.copy(), SAMPLE_RATE)


def benchmark_dataloader(num_workers, length=32):
    dataset = AugmentationBenchmarkDataset(audio, combined_augmentation, length)
    loader = DataLoader(dataset, batch_size=8, num_workers=num_workers,
                        persistent_workers=num_workers > 0)
    start = time.perf_counter()
    examples = 0
    for batch in loader:
        examples += len(batch)
    elapsed = time.perf_counter() - start
    return {'workers': num_workers, 'examples_per_second': examples / elapsed,
            'seconds': elapsed}

pd.DataFrame([benchmark_dataloader(workers) for workers in (0, 2, 4)])

In [ ]:
import torch 
model = torch.jit.load("/data/nils/repos/AFTER/pretrained/instruments.ts").cuda()
wav = torch.randn(16, 1, 131072).cuda()
for i in range(2):
    model.encode(wav)


In [ ]:
import time
st = time.time()


with torch.inference_mode():
    for i in range(20):
        model.encode(wav)

end = time.time()-st

print(131072/44100)
print(end/20)
